# TP : Assistant RAG intelligent avec LangChain, ChromaDB, Ollama et Gradio

## Objectif

L'objectif de ce TP est de concevoir un assistant conversationnel basé sur l'architecture **RAG (Retrieval-Augmented Generation)**.

L'application permet de :

- charger automatiquement plusieurs documents PDF ;
- indexer leur contenu dans une base vectorielle ChromaDB ;
- rechercher les passages les plus pertinents ;
- générer des réponses grâce à un modèle LLM exécuté localement avec Ollama ;
- résumer un TP spécifique ;
- répondre à des questions simples et complexes ;
- afficher les sources utilisées ;
- proposer une interface Web avec Gradio.

In [ ]:
import os
import gradio as gr

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama

In [ ]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

# 1. Chargement des documents PDF

Cette étape consiste à charger automatiquement tous les fichiers PDF présents dans le dossier **documents/**.

Chaque document est ensuite enrichi avec des métadonnées (nom du fichier, numéro de page) afin de pouvoir citer précisément les sources utilisées lors des réponses.

In [ ]:
DOSSIER_PDF = "documents"

documents = []

for fichier in os.listdir(DOSSIER_PDF):
    if fichier.lower().endswith(".pdf"):
        chemin = os.path.join(DOSSIER_PDF, fichier)
        print("Chargement :", fichier)

        loader = PyPDFLoader(chemin)
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = fichier

        documents.extend(docs)

print("Nombre total de pages :", len(documents))

# 2. Découpage des documents

Les documents sont découpés en petits morceaux (*chunks*) grâce à **RecursiveCharacterTextSplitter**.

Ce découpage améliore la qualité de la recherche vectorielle tout en conservant suffisamment de contexte pour le modèle de langage.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Nombre total de chunks :", len(chunks))

# 3. Création de la base vectorielle

Chaque chunk est transformé en vecteur grâce au modèle d'embedding **sentence-transformers/all-MiniLM-L6-v2**.

Les vecteurs sont ensuite stockés dans **ChromaDB**, qui servira de base documentaire du système RAG.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="db_tp"
)

retriever = vectordb.as_retriever(
    search_kwargs={"k": 4}
)

print("Base vectorielle créée.")

# 4. Recherche documentaire

Le retriever recherche automatiquement les passages les plus pertinents dans la base vectorielle.

Le système peut également limiter la recherche à un TP spécifique (TP1, TP2, TP3 ou TP4) lorsque celui-ci est mentionné dans la question.

In [ ]:
def lister_les_tp():
    fichiers = []

    for fichier in os.listdir("documents"):
        if fichier.lower().endswith(".pdf"):
            fichiers.append(fichier)

    fichiers = sorted(fichiers)

    resultat = "Les TP disponibles sont :\n\n"

    for i, fichier in enumerate(fichiers, start=1):
        resultat += f"{i}. {fichier}\n"

    return resultat

In [ ]:
def detecter_tp(question):
    question_min = question.lower()

    if "tp1" in question_min:
        return "tp1"
    if "tp2" in question_min:
        return "tp2"
    if "tp3" in question_min:
        return "tp3"
    if "tp4" in question_min:
        return "tp4"

    return None


def rechercher_dans_tp(question):
    tp_filtre = detecter_tp(question)

    if tp_filtre:
        docs = [
            doc for doc in chunks
            if tp_filtre in doc.metadata.get("source", "").lower()
        ][:6]
    else:
        docs = retriever.invoke(question)

    contexte = "\n\n".join([doc.page_content for doc in docs])

    sources = []
    for doc in docs:
        source = doc.metadata.get("source", "Inconnu")
        page = doc.metadata.get("page", "?")
        sources.append(f"- {source} (page {page})")

    return {
        "question": question,
        "contexte": contexte,
        "sources": "\n".join(dict.fromkeys(sources))
    }

# 5. Résumé automatique d'un TP

Une fonction spécifique permet de produire un résumé structuré d'un TP.

Le résumé est généré uniquement à partir des passages retrouvés dans les documents concernés.

In [ ]:
def resumer_tp(question):
    resultat = rechercher_dans_tp(question)

    prompt = f"""
Tu es un assistant pédagogique spécialisé dans les TP d'IA.

Ta tâche est de résumer le document demandé.

Utilise UNIQUEMENT le contexte fourni.
N'invente rien.

Question :
{question}

Contexte :
{resultat["contexte"]}

Consignes :
- Réponds en français.
- Fais un résumé clair et structuré.
- Si la question demande "en 10 lignes", fais exactement 10 points numérotés.
- Ne parle pas des documents qui ne sont pas dans les sources.
"""

    reponse = llm.invoke(prompt)

    return f"""{reponse.content}

------------------------
Sources utilisées :

{resultat["sources"]}
"""

# 6. Assistant conversationnel RAG

L'assistant combine les résultats du retriever avec le modèle de langage Ollama.

Le prompt impose plusieurs règles :

- répondre uniquement à partir des documents ;
- effectuer des synthèses lorsque plusieurs documents sont pertinents ;
- éviter les hallucinations ;
- citer les sources utilisées.

In [ ]:
def chat_rag(question):

    question_min = question.lower()

    if "tp disponibles" in question_min or ("liste" in question_min and "tp" in question_min):
        return lister_les_tp()

    if "résume" in question_min or "resume" in question_min:
        return resumer_tp(question)

    resultat = rechercher_dans_tp(question)

    prompt = f"""
Tu es un assistant RAG spécialisé dans les TP d'IA.

Règles :

Tu es un assistant RAG spécialisé sur les TP d'Intelligence Artificielle.

Tu dois répondre UNIQUEMENT à partir du contexte fourni.

RÈGLES OBLIGATOIRES :

1. Réponds uniquement à partir des informations présentes dans le contexte.

2. Si plusieurs documents contiennent des informations complémentaires, tu peux les synthétiser, les comparer et les mettre en relation.

3. Tu peux reformuler, organiser et résumer les informations présentes dans les documents afin de produire une réponse claire et pédagogique.

4. Tu ne dois jamais utiliser de connaissances extérieures aux documents.

5. Tu ne dois jamais inventer d'informations ni faire d'hypothèses.

6. N'utilise jamais des expressions telles que :
- "il est possible que..."
- "on peut imaginer..."
- "probablement..."
- "pourrait être..."
- "en général..."
- "habituellement..."

7. Si les documents ne contiennent pas suffisamment d'informations pour répondre, réponds exactement :

"Je ne trouve pas suffisamment d'informations dans les documents pour répondre à cette question."

8. Si la question demande une comparaison ou une synthèse, réalise-la uniquement à partir des informations présentes dans les documents.

9. Réponds toujours en français.

10. Structure tes réponses avec des titres ou des puces lorsque cela améliore la lisibilité.

11. À la fin de chaque réponse, cite uniquement les documents réellement utilisés.


Question :
{question}

Contexte :
{resultat["contexte"]}

Réponse :
"""

    reponse = llm.invoke(prompt)

    return f"""{reponse.content}

------------------------
Sources utilisées :

{resultat["sources"]}
"""

In [ ]:
print(chat_rag("Quels sont les TP disponibles ?"))

In [ ]:
print(chat_rag("Résume TP1_Ingenierie_des_prompts.pdf en 10 lignes"))

# 7. Tests de l'assistant

Plusieurs questions sont utilisées afin de vérifier le fonctionnement du système :

- liste des TP disponibles ;
- résumé d'un TP ;
- questions sur le contenu des TP ;
- synthèses entre plusieurs documents ;
- questions complexes nécessitant une recherche documentaire.

In [ ]:
questions_simples = [
    "Quels sont les TP disponibles ?",
    "Résume TP1_Ingenierie_des_prompts.pdf en 10 lignes",
    "Résume TP2_Agents_avec_LangChain.pdf",
    "Résume TP3_Agent_RAG.pdf",
    "Résume TP4_Agent MCP.pdf",
    "Qu'est-ce que le Prompt Engineering ?",
    "Qu'est-ce qu'un système RAG ?",
    "Qu'est-ce que LangChain ?",
    "Qu'est-ce que LangGraph ?",
    "Quels TP parlent des agents ?"
]

questions_complexes = [
    "En t'appuyant sur les TP2 et TP3, explique le rôle du LLM et du retriever dans un système RAG.",
    "En t'appuyant sur les TP2, TP3 et TP4, présente les principales utilisations de LangChain et de LangGraph observées dans les documents.",
    "En t'appuyant sur les TP2 et TP4, quelles différences observes-tu entre un agent simple et un agent utilisant plusieurs outils ou serveurs MCP ?",
    "Explique les étapes de construction d'un système RAG.",
    "Quels TP utilisent MCP et pourquoi ?",
    "Quels TP utilisent des tools ?",
    "Quels concepts sont communs à tous les TP ?",
    "Fais un tableau comparatif des TP.",
    "Donne un plan de révision basé sur tous les TP.",
    "Quels sont les prérequis pour comprendre tous les TP ?"
]

# 8. Interface utilisateur

Une interface Web est réalisée avec **Gradio**.

L'utilisateur peut :

- poser librement ses questions ;
- consulter une liste de questions d'exemple ;
- obtenir une réponse argumentée ;
- visualiser les documents et les sources utilisés.

In [ ]:
def repondre(message, history):
    return chat_rag(message)


with gr.Blocks() as interface:
    gr.Markdown("# 🎓 Assistant RAG des TP d'IA")
    gr.Markdown("Assistant documentaire basé sur LangChain, ChromaDB, Ollama et Gradio.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 📚 Informations")
            gr.Markdown("""
**Documents indexés :** TP1, TP2, TP3, TP4  
**Technologies :** LangChain, ChromaDB, Ollama, Gradio  
**Objectif :** interroger les documents de TP
""")

            gr.Markdown("## 🟢 Questions simples")
            for q in questions_simples:
                gr.Markdown(f"- {q}")

            gr.Markdown("## 🔴 Questions complexes")
            for q in questions_complexes:
                gr.Markdown(f"- {q}")

        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=repondre,
                title="💬 Chat avec les documents",
                description="Pose une question sur les TP indexés.",
                examples=questions_simples + questions_complexes
            )

interface.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


c:\Users\hp\Desktop\langchainPrompt\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\hp\Desktop\langchainPrompt\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\hp\Desktop\langchainPrompt\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\hp\Desktop\langchainPrompt\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  retu

# Conclusion

Ce TP a permis de développer un assistant documentaire intelligent reposant sur l'architecture RAG.

Les principales fonctionnalités obtenues sont :

- chargement automatique des documents PDF ;
- indexation vectorielle avec ChromaDB ;
- recherche sémantique des passages pertinents ;
- génération de réponses avec Ollama ;
- résumé automatique des TP ;
- synthèse de plusieurs documents ;
- affichage des sources utilisées ;
- interface graphique avec Gradio.

Ce projet illustre la mise en œuvre complète d'un assistant RAG moderne capable d'exploiter efficacement une base documentaire locale.